# SDA-DSC-212 — Time Series Analysis and Forecasting
## Lab 7 — End-to-End Selection and the Decision Record

**Duration** 50 minutes · **Work in pairs** · `git checkout lab7-start`

> The one rule for this whole course: every feature, every split and every scaler
> must be computable using only data that existed at the forecast origin.

### Tasks

| min | task |
|---|---|
| 8 | Wire the pipeline end to end from a **committed config** and confirm a clean run reproduces the Lab 6 backtest table. Print `MATCH`. Then read the inherited selection step and find the **fault** in it. |
| 10 | Three constraint scenarios with explicit weights over accuracy / explainability / latency / maintenance. Build the candidate profile table and run `recommend.score` for each. |
| 10 | Observe that the champion **changes** across scenarios. Record which constraint flipped each decision, and use `flip_conditions` to state what would have to change for the runner-up to win. |
| 8 | Generate the reserve brief with `pipeline.brief.reserve_brief`, then critique it as a control-room engineer would. Rewrite the jargon version in decision language. |
| 9 | Write `DECISION_RECORD.md`: the decision and its cost function, the candidates, the constraints and weights, the recommendation with its DM evidence, the flip-conditions, and the monitoring contract. |
| 5 | Commit. |

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})

DATA = "../data"

### Task 1 — the config, the clean run, and the selection step you inherited

**Reproducibility is what makes the Lab 6 report evidence rather than anecdote.** Three things
buy it, and all three are cheap: **fixed splits** (dates in a file, not in a head), **fixed
seeds** (every LightGBM in `src/tayyar` is seeded at 212), and **pinned dependencies** (a
LightGBM minor version can move the third decimal). A number nobody else can regenerate is not
a result; it is a claim.

The test is blunt: run the whole pipeline from the committed config in a clean kernel and
check that it reproduces `reference_results.json` exactly. If it prints `MATCH`, the report
can be defended. If it does not, nothing downstream of it means anything.

In [ ]:
import json

CONFIG = {
    "data": {"demand": f"{DATA}/ksa_grid_demand.csv",
             "calendar": f"{DATA}/ksa_calendar.csv"},
    "features": {"source_columns": ["demand_mw", "temp_c", "is_weekend",
                                    "is_ramadan", "is_eid", "is_national_day"]},
    "split": {"cut_train": "2023-09-30 23:00", "cut_calibration": "2023-10-31 23:00"},
    "backtest": {"horizon": 24, "n_origins": 60, "step": 24,
                 "initial_train": 24 * 365, "window": "expanding", "m": 24},
    "models": {"point": "DirectLGBMForecaster(horizon=24)",
               "quantile": "QuantileLGBM(quantiles=(0.05, 0.50, 0.95), horizon=24)",
               "baseline": "seasonal_naive(m=24)"},
    "interval": {"method": "CQR", "alpha": 0.10, "calibration_window": "2023-10"},
    "seeds": {"lightgbm": 212, "numpy": 212},
    "refit": {"cadence": "monthly", "boundary": "2023-09-30 23:00"},
}
print(json.dumps(CONFIG, indent=2))

In [ ]:
# TODO (8 min)
#
#   1. Write run_pipeline(CONFIG): load -> build_features -> BacktestConfig ->
#      fit DirectLGBMForecaster and QuantileLGBM on data ending at the config's
#      cut_train -> backtest the three families at the config's origins.
#      NOTHING may come from the caller's scope. That is what "config-driven" means.
#   2. Print the installed versions of pandas, numpy, lightgbm, scikit-learn,
#      statsmodels and scipy alongside the result.
#   3. Compare your summary table with reference_results.json["m6"]["summary"] and
#      the two Diebold-Mariano results. Print MATCH or MISMATCH — and mean it: if it
#      mismatches, nothing downstream of it is worth writing down.
#   4. Then run the inherited selection cell below and find the fault in it.
#
# HINT: reproducibility is fixed splits + fixed seeds + pinned versions. If your
#       numbers move between runs, look for a split date typed into a cell, an
#       unseeded model, or a dropna() that silently depends on execution order.

In [ ]:
# The backtest table Lab 6 produced, loaded from the committed reference run so that
# the inherited selection step below runs before you have finished rebuilding it.
# Replace it with YOUR pipeline's output as soon as Task 1 prints MATCH.
import json

MET = ["MAE", "RMSE", "MAPE_%", "WAPE_%", "MASE", "bias"]
bt = pd.DataFrame(json.load(open("../reference_results.json"))["m6"]["summary"])
summary = bt.set_index("model")[MET].sort_values("MASE")

In [ ]:
# ---------------------------------------------------------------------------
# The selection step you inherited
#
#   "Backtest reproduces. Champion selected. Ready to promote."
#
# It runs, it uses the library function, and the model it names is genuinely the
# most accurate one in the table. It is still the wrong answer.
# ---------------------------------------------------------------------------
from tayyar.pipeline.select import champion

winner = champion(bt, metric="MASE")
print(summary.to_string())
print()
print(f"CHAMPION: {winner}")
print(f"MASE {summary.loc[winner, 'MASE']:.3f} — lowest in the table. Promoting to production.")

In [ ]:
# TODO (part of Task 1) — the selection step above is wrong. It is not wrong
# because it picked the wrong row; it is wrong because of the QUESTION it asks.
#
#   1. What is champion() actually optimising? Read pipeline/select.py.
#   2. Lab 6 put the top two models at p = 0.058 on a Diebold-Mariano test. What does
#      a sort do with that information?
#   3. List everything the winning model needs AT SERVE TIME, at 23:00, to produce
#      tomorrow's forecast. Which of those things does a regulated batch publication
#      job actually have?
#   4. What does the decision downstream consume — a number, or a distribution?
#
# Write the fault in one sentence at the top of DECISION_RECORD.md before you fix it.
#
# HINT: "the most accurate model" and "the model we should run" are different
#       questions, and only one of them can be answered by sorting a column.

### Task 2 — three use cases, three weightings, one candidate set

**Selection is a function of the use case, not a position on a leaderboard.** The same three
models, ranked for three different customers, produce three different answers — and none of
the rankings is wrong.

The six constraints that legitimately override accuracy:

1. **Deadline and latency.** A forecast that misses the publication window has an error of
   infinity. A model that trains in four hours cannot serve a 15-minute re-run.
2. **Explainability.** A regulator, or an operator at 02:00, has to be able to interrogate the
   forecast. "Feature importance says lag-3" is not an answer to "why is it high tonight?".
3. **Cost asymmetry.** Under-forecasting load costs far more than over-forecasting it — the
   first buys emergency generation, the second buys idle spinning reserve. A symmetric MAE
   objective encodes a symmetry the business does not have. Optimise a quantile or an
   asymmetric loss and select on it.
4. **Maintenance and total cost of ownership.** One model per operating area, per horizon, per
   quantile is a pipeline somebody owns forever. Multiply by four areas before you choose.
5. **Serve-time data availability.** A feature you cannot compute at 23:00 is not a feature.
   No day-ahead temperature feed means no SARIMAX-with-temperature and no weather columns —
   whatever the backtest said.
6. **Adoption.** The winning model is the one people will actually use at 02:00 during an
   unusual event. A model the control room overrides every time it matters has zero skill in
   production regardless of its MASE.

The rule that follows: **the simplest model meeting the constraints wins, and complexity is
justified only by a DM-significant, decision-relevant gain.**

In [ ]:
from tayyar.pipeline.recommend import score, flip_conditions

# TODO (10 min)
#
#   1. Build `profiles`: one row per candidate family, columns accuracy /
#      explainability / latency / maintenance, scored 0-1, higher is better.
#      Derive them from the trade-off space, not from taste:
#        seasonal-naive  accuracy low, explainability total, no interval,
#                        latency trivial, maintenance none
#        ETS             medium / high / analytic interval / very fast / low burden
#        SARIMAX + temp  medium-high / high, a component story / native analytic
#                        interval / moderate, slow at large m / medium burden
#        LightGBM + CQR  high / medium, importance only / calibrated and sharp /
#                        fast to train but needs a feature store / higher, a pipeline
#      Add a GLOBAL LightGBM row (one model across the four areas).
#   2. Build `facts`: the hard, non-negotiable columns — needs_feature_store,
#      needs_temp_forecast, models_to_own_4_areas, interval type. These become
#      FILTERS. A constraint you can score away is not a constraint.
#   3. Three scenarios, with the weights written down BEFORE you look at the result:
#        A Regulator          0.3 / 0.4 / 0.2 / 0.1  + hard: no feature store
#        B Internal dispatch  0.6 / 0.1 / 0.2 / 0.1
#        C Four areas         0.3 / 0.1 / 0.2 / 0.4
#      Apply the hard filter, then score(elig, weights), and print the champion.
#
# HINT: if the same model wins all three scenarios, either your weights are barely
#       different or your hard filters are not being applied. Check both.

### Task 3 — the champion changes, and you must say what changed it

A recommendation that does not state its own flip-conditions is not a recommendation, it is a
preference. `flip_conditions` answers one question precisely: **how much would have to move,
and in which column, for the runner-up to win?** If the answer is "0.02 of explainability",
the decision is a coin toss dressed as analysis and you must say so.

In [ ]:
# TODO (10 min)
#
#   1. Tabulate, per scenario: champion, weighted score, runner-up, the GAP between
#      them, and the constraint you judge to have been decisive.
#   2. flip_conditions(ranked[name], weights) for each scenario. Print every line.
#   3. Then the question that matters: which of these three decisions is real, and
#      which is a tie you have dressed up as a decision? Look at the gaps.
#
# HINT: a 0.005 gap between two candidates is not a champion. Say so in the record,
#       and apply the same rule Lab 6 applied to p = 0.058 — take the simpler one.

### Task 4 — the brief, and what a control-room engineer does with it

The forecast is finished. The **communication** is not, and it is where forecasts are most
often rejected. Lead with the decision, not the model.

In [ ]:
from tayyar.pipeline.brief import reserve_brief

# TODO (8 min)
#
#   1. Calibrate CQR on the October window from CONFIG, then build a one-day forecast
#      frame for the last origin with columns q05 / q50 / q95 and a DatetimeIndex.
#   2. brief = reserve_brief(fc, coverage=..., n_origins=..., weakness=..., fallback=...)
#      The weakness and fallback arguments are not decoration. Write them as if the
#      person reading them will be woken at 02:00 by exactly the failure you name.
#   3. Print the brief. Then critique it as a control-room engineer:
#        - what is the FIRST sentence, and is it a decision or a description?
#        - which numbers would you act on, and which are for the modeller?
#        - what does it tell you to do when it is wrong?
#   4. Write the jargon version and the decision version side by side:
#        "the p95 is X MW, MASE is 0.455, marginal coverage is 0.90"
#        vs
#        "set spinning reserve to X MW; expected peak is Y at 15:00; the gap is the
#         honest uncertainty and it is widest on hot days"
#
# HINT: every number in the jargon version is correct. That is the problem — none of
#       them is a decision, and a correct sentence nobody can act on is a failed one.

### Task 5 — `DECISION_RECORD.md`

Seven sections, in this order. The order is the argument.

**1. The decision and its cost function.** *How much spinning reserve to hold for each hour of
tomorrow.* Under-forecasting buys emergency generation at a large multiple of the marginal
cost, and in the tail buys load shedding; over-forecasting buys idle capacity at the marginal
cost of holding it. The costs are asymmetric, so the object being decided is a **quantile**,
not a mean, and the model is selected on that basis.

**2. The candidates.** Seasonal-naive, ETS, SARIMAX + temperature, LightGBM + CQR (per area),
LightGBM + CQR (global). With the profile table.

**3. The constraints applied**, hard filters separated from weighted criteria — publication
deadline, explainability to the regulator, cost asymmetry, maintenance across four areas,
serve-time data availability (feature store, day-ahead temperature feed), and adoption by the
control room.

**4. The stated weights**, per scenario, written down before the scores were computed.

**5. The recommendation and its evidence.** The champion per scenario, with the Lab 6
Diebold-Mariano results attached: significant against the baseline (dm = -6.75, p < 0.001),
**not** significant between the two LightGBM variants (p = 0.058) — which is why the quantile
model is preferred over the marginally more accurate direct model in scenario B, and why
scenario C's 0.005 gap is reported as a tie.

**6. The flip-conditions.** Verbatim from `flip_conditions`, per scenario, plus the operational
ones: no day-ahead temperature feed removes SARIMAX + temperature; a fifth operating area
moves scenario C further towards the global model; a regulator accepting a documented
importance analysis re-admits LightGBM to scenario A.

**7. The monitoring contract**, which is the part that makes the recommendation survive
contact with production:

| what | how often | threshold | action |
|---|---|---|---|
| rolling MASE, 7-day and 28-day | weekly | 28-day MASE > 0.60 | investigate; > 0.77 (the baseline's own score) means fall back |
| empirical coverage of the 90% band | weekly | outside 0.85-0.95 for two consecutive weeks | re-estimate q̂ on the last 30 days |
| conditional coverage on the 12:00-18:00 block | weekly | below 0.85 | widen the peak-hour band; consider adaptive conformal |
| feature drift (temperature, weekly mean level) | weekly | new maximum outside the training range | expect extrapolation failure; trees cannot forecast a record high |
| retrain | monthly, and on any threshold breach | — | full refit, then re-run this backtest before promotion |
| champion / challenger | continuous | challenger beats champion on a DM test at 5% over 60 origins | promote; otherwise keep the incumbent |

**Say what the recommendation cannot do.** This backtest window contains no Eid, no summer
heatwave and no record peak. That absence is a finding, and naming it is what earns adoption:
a report that names the Eid weeks and ships a fallback should outscore a silent, marginally
sharper submission, because the second one will be overridden the first time it is wrong and
then ignored forever after. Candour is not a weakness in a recommendation; it is the thing
that makes it usable at 02:00.

### Mini-quiz

Answer these in your pair before you look at the next cell.

1. Is "lowest MASE" always the right selection?
2. Which family fits "covariates plus a regulator demanding explainability"?
3. What must a recommendation include besides the champion?
4. Why present a quantile rather than a point forecast for reserve sizing?
5. What makes selection evidence reproducible?

### Task 6 — commit

```
git add -A && git commit -m "feat(pipeline): constraint-weighted selection, reserve brief and decision record"
```

The two files that leave this course are `BACKTEST.md` (the evidence) and
`DECISION_RECORD.md` (the decision). The notebooks are the working; those two are the work.